# 4. LLM-режим команды Graph-EVT-agent

**Цель:** запустить детерминированный `GraphEVTPipeline` и попросить LLM проверить результат относительно синтетической разметки. Для этого учебного набора `is_extreme` задаёт известную границу события и используется только для выбора чистого baseline и последующей оценки; сама метка не передаётся численному детектору.

Для живого запуска задайте `MISTRAL_API_KEY` в окружении или введите его в скрытом prompt. Ключ не записывается в ноутбук или артефакты. Использование истинной метки для определения baseline здесь oracle-assisted и не подходит для неразмеченных реальных данных.


In [14]:
from pathlib import Path
import os
from getpass import getpass
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig
from graph_evt_agent.agents import EVTAgentTeam, MistralAPIError, MistralClient

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/fractal_extreme_series.csv").exists())
DATA = ROOT / "data/generated/fractal_extreme_series.csv"
SEED = 42
df = pd.read_csv(DATA)
x = df["value"].to_numpy()
event_mask = df["is_extreme"].to_numpy(dtype=bool)
assert np.isfinite(x).all() and x.ndim == 1 and event_mask.shape == x.shape
assert event_mask.any(), "Generated dataset must contain a labeled event"
event_start = int(np.flatnonzero(event_mask)[0])
event_end = int(np.flatnonzero(event_mask)[-1])


In [15]:
api_key = os.getenv("MISTRAL_API_KEY") or getpass("Mistral API key (input hidden): ").strip()
MISTRAL_READY = False
mistral_client = None

if api_key:
    try:
        mistral_client = MistralClient(
            api_key=api_key,
            model="ministral-3b-2512",
            max_retries=0,
        )
        check_reply = mistral_client.complete(
            "Reply exactly with MISTRAL_OK.",
            "Connection check. Reply exactly with MISTRAL_OK.",
        )
        MISTRAL_READY = bool(check_reply.strip())
        print("Mistral API check passed." if MISTRAL_READY else "Mistral returned an empty response.")
    except MistralAPIError as error:
        error_text = str(error).lower()
        if "429" in error_text or "rate_limit" in error_text or "rate limit" in error_text:
            print("Mistral rate limit (HTTP 429). Check your account usage and rate limits, wait, then rerun this cell.")
        else:
            print(f"Mistral API check failed: {error}")
    except Exception as error:
        print(f"Mistral API check failed ({type(error).__name__}).")
else:
    print("Mistral API check skipped: no key was provided.")


Mistral API check passed.


In [16]:
# In this synthetic dataset, use the known event onset to keep the EVT baseline clean.
# The label is used for evaluation and baseline selection, not as detector input.
BASELINE_SIZE = event_start
pipeline = GraphEVTPipeline(
    EVTConfig(baseline_size=BASELINE_SIZE, tail_quantile=0.95, alarm_probability=0.999, persistence=3),
    graph=GraphConfig(random_state=SEED),
    input_config=InputConfig(missing="error"),
)
numeric_result = pipeline.run_detailed(x)
detection = numeric_result.detection
alarm_index = detection.time_index
detected_in_event = alarm_index is not None and bool(event_mask[alarm_index])
detection_delay = alarm_index - event_start if detected_in_event else None

pd.DataFrame([{
    "labeled_event_start": event_start,
    "labeled_event_end": event_end,
    "baseline_size": BASELINE_SIZE,
    "detected": detection.detected,
    "alarm_index": alarm_index,
    "detected_in_labeled_event": detected_in_event,
    "detection_delay_samples": detection_delay,
    "alarm_threshold": detection.alarm_threshold,
}])


,labeled_event_start,labeled_event_end,baseline_size,detected,alarm_index,detected_in_labeled_event,detection_delay_samples,alarm_threshold
0,604,678,604,True,619,True,15,3.332388


In [17]:
if MISTRAL_READY:
    team = EVTAgentTeam(pipeline, mistral_client)
    try:
        agent_report = team.run(
            x,
            task=(
                "Проверь результат EVT-детекции относительно синтетической разметки: "
                f"истинный интервал события [{event_start}, {event_end}], "
                f"detected={detection.detected}, alarm_index={alarm_index}, "
                f"delay_samples={detection_delay}. Укажи явно, попало ли срабатывание "
                "в размеченное событие; не называй отсутствие срабатывания доказательством "
                "отсутствия аномалии."
            ),
        )
        print(agent_report.final_report)
    except MistralAPIError as error:
        error_text = str(error).lower()
        if "429" in error_text or "rate_limit" in error_text or "rate limit" in error_text:
            print("Mistral rate limit during review. Check account usage/rate limits, wait, then rerun this cell.")
        else:
            print(f"LLM review failed: {error}")
    except Exception as error:
        print(f"LLM review failed ({type(error).__name__}).")
else:
    print("LLM review skipped; the numeric pipeline result is still available.")


### **Краткий итог по анализу данных**

#### **1. Структура данных**
- **Тип:** Унивариантный (1-D) одноканальный временной ряд (`n_channels: 1`, `n_time: 1500`).
- **Статистическая редкость:** Формально соответствует ожидаемой структуре, но **отсутствуют временные метки** (`has_timestamps: false`), что ограничивает анализ.

#### **2. Временная ось**
- **Проблемы:**
  - Нет явных временных меток или регулярного интервала (`sampling_interval: null`).
  - **Ограничение:** Без временных данных невозможно определить:
    - Начало/конец временного интервала.
    - Регулярность/нерегулярность шагов.
    - Пропуски по времени (только проверены пропуски значений, `missing_per_channel: [0]`).
- **Рекомендация:** Добавьте временные метки или регулярный интервал для корректного анализа.

#### **3. Пропуски данных**
- **Отсутствуют** (`missing_per_channel: [0]`), но без временных меток нельзя проверить пропуски по времени.

#### **4. Объём данных**
- **1500 точек** (одноканальный временной ряд).



`EVTAgentTeam` сначала вызывает тот же детерминированный pipeline. Агентам передаются агрегированные evidence-поля, а не право переписать вычисления. Для 1-D входа графовый и локализационный агенты корректно пропускаются. Текст LLM является интерпретацией и должен проверяться по `numeric_result`.